# Patrón Command (Comando)

## 1) ¿Qué problema resuelve?
Cuando quieres representar una acción (una “operación”) como un **objeto** para poder:
- **encolarla** (queues)
- **reintentarla** (retry/backoff)
- **programarla** (scheduling)
- **registrarla** (auditoría/log)
- **deshacerla** (undo) o compensarla

En lugar de ejecutar directamente:
- `payment_service.charge(order_id, amount)`
encapsulas la intención:
- `ChargePaymentCommand(order_id, amount).execute()`

---

## 2) Idea central
**Encapsular una solicitud/acción como objeto**, de manera que:
- el *invocador* (quien “pide” ejecutar) no depende del *receptor* (quien “hace” el trabajo),
- la acción se vuelve **transportable**: se puede guardar, reintentar, enviar a una cola, ejecutar luego.

El contrato típico es un método:
- `execute()`

Opcionalmente (para undo):
- `undo()` o *compensating action*.

---

## 3) Componentes del patrón (GoF)
### Command (Interfaz)
- Define la operación uniforme:
  - `execute()`

### ConcreteCommand (Comando concreto)
- Implementa `execute()`.
- Contiene **los datos necesarios** para ejecutar la acción.
- Normalmente mantiene una referencia al **Receiver**.

### Receiver (Receptor)
- Es quien realiza el trabajo real.
- Ej: `NotificationSender`, `PaymentService`, `InventoryService`.

### Invoker (Invocador)
- Dispara la ejecución del comando.
- Puede aplicar política: reintentos, logs, rate limits.
- Ej: `CommandRunner`, `Scheduler`, `QueueWorker`.

### Client (Cliente)
- Construye el comando y lo “configura” (inyecta receiver + params).
- Pasa el comando al invoker.

---

## 4) Flujo de ejecución
1. **Client** crea `ConcreteCommand(receiver, params...)`.
2. **Invoker** recibe el comando (ahora o luego).
3. **Invoker** llama `command.execute()`.
4. **Command** delega al **Receiver** para hacer el trabajo.

---

## 5) ¿Cuándo usar Command?
- **Jobs asíncronos**:
  - enviar email
  - procesar pagos
  - generar reportes
- **Retries**:
  - APIs externas fallan de forma temporal
  - circuit breaker/backoff
- **Workflows**:
  - checkout: charge → reserve → ship
- **Auditoría**:
  - registrar qué acciones se ejecutaron y con qué parámetros
- **Sagas / compensación**:
  - si falló un paso posterior, ejecutar “compensación” (undo lógico)

---

## 6) Ventajas
- **Desacoplamiento**: invoker no necesita conocer el receiver.
- **Reintentos centralizados**: la política vive en `CommandRunner`, no en cada servicio.
- **Composición**: se pueden encadenar comandos (pipeline/workflow).
- **Observabilidad**: fácil registrar “qué comando se ejecutó” y sus parámetros.
- **Testabilidad**: pruebas enfocadas por comando; puedes mockear receivers.

---

## 7) Trade-offs y riesgos
### 7.1 Explosión de clases
- Cada acción puede convertirse en una clase nueva.
- Mitigación: agrupar comandos por módulo o usar handlers con nombres claros.

### 7.2 Idempotencia (clave en sistemas reales)
En colas y retries, es común el modelo **at-least-once delivery**:
- un comando puede ejecutarse más de una vez.
- Si no es idempotente, puedes cobrar dos veces o enviar dos notificaciones.

Solución típica:
- `request_id` / `command_id`
- tabla/registro de “comandos procesados”
- checks antes de aplicar efectos.

### 7.3 Undo no siempre existe
- Muchas operaciones no se pueden “deshacer” literalmente.
- Se usa **compensación** (Saga):
  - `ChargePayment` compensa con `RefundPayment`
  - `ReserveInventory` compensa con `ReleaseInventory`

### 7.4 Estado y persistencia
- Si el comando debe ejecutarse más tarde, debes persistir:
  - parámetros
  - contexto mínimo
  - correlación/traza

---

## 8) Buenas prácticas (checklist)
- Definir interfaz clara `execute()`.
- Comandos deben ser:
  - pequeños
  - con una responsabilidad
  - explícitos en parámetros
- Si hay reintentos:
  - incluir `request_id` y diseñar idempotencia
  - aplicar backoff y límite de reintentos
- Logging estructurado:
  - `command_name`, `command_id`, `status`, `error_reason`
- Separar:
  - **qué** se quiere hacer (Command)
  - **cómo y cuándo** se ejecuta (Invoker/Runner/Queue)

---

## 9) Comparación rápida con patrones cercanos
### Command vs Strategy
- **Strategy**: elegir un algoritmo (cómo se calcula algo).
- **Command**: encapsular una acción (hacer algo) como objeto.

### Command vs Observer
- **Observer**: reacción a eventos (fan-out de listeners).
- **Command**: ejecutar acciones controladas (cola, retry, scheduling).

---

## 10) Ejemplo conceptual
- Receiver: `NotificationSender.send(notification)`
- Command: `SendNotificationCommand(sender, notification, request_id)`
- Invoker: `CommandRunner(max_retries=3).run(command)`

Esto permite:
- reintentar si el provider falla
- encolar el comando
- registrar ejecución
- evitar duplicados con `request_id`

In [7]:
class Command:
    def execute() -> None: ...
    def undo() -> None: ...

In [8]:
class Luz:
    def __init__(self) -> None:
        self.is_on = False
    
    def on(self) -> None:
        self.is_on = True
        print('Luz prendida!')
    
    def off(self) -> None:
        self.is_on = False
        print('Luz apagada!')

In [9]:
class Ventilador:
    def __init__(self):
        self.velocidad = 0 # 0 (apagado), 1, 2
    
    def set_velocidad(self, velocidad: int) -> None:
        self.velocidad = velocidad
        print(f'Velocidad del ventilador: {self.velocidad}')

In [10]:
from dataclasses import dataclass

In [11]:
@dataclass
class PrenderLuzCommand:
    luz: Luz

    def execute(self):
        self.luz.on()
    
    def undo(self):
        self.luz.off()

@dataclass
class ApagarLuzCommand:
    luz: Luz

    def execute(self):
        self.luz.off()
    
    def undo(self):
        self.luz.on()

In [12]:
@dataclass
class SetVelocidadVentiladorCommand:
    ventilador: Ventilador
    velocidad: int
    velocidad_prev: int = 0

    def execute(self):
        self.velocidad_prev = self.ventilador.velocidad
        self.ventilador.set_velocidad(self.velocidad)
    
    def undo(self):
        self.ventilador.set_velocidad(self.velocidad_prev)

In [ ]:
from typing import Optional

In [13]:
class ControlRemoto:
    def __init__(self):
        self.ultimo_comando: Optional[Command] = None
    
    def presionar(self, command: Command) -> None:
        command.execute()
        self.ultimo_comando = command
    
    def presionar_undo(self) -> None:
        if self.ultimo_comando:
            self.ultimo_comando.undo()
        else:
            print('Nada que deshacer')

In [17]:
luz_sala = Luz()
ventilador_sala = Ventilador()

luz_cuarto = Luz()

remoto = ControlRemoto()

In [22]:
remoto.presionar(PrenderLuzCommand(luz_sala))
remoto.presionar(SetVelocidadVentiladorCommand(ventilador_sala, 1))
remoto.presionar(PrenderLuzCommand(luz_cuarto))
remoto.presionar(SetVelocidadVentiladorCommand(ventilador_sala, 2))


remoto.presionar_undo()
remoto.presionar_undo()

Luz prendida!
Velocidad del ventilador: 1
Luz prendida!
Velocidad del ventilador: 2
Velocidad del ventilador: 1
Velocidad del ventilador: 1
